# 05. 데코레이터 (decorator)

함수를 감싸서 동작을 추가하는 패턴. 로깅, 타이밍, 캐싱, 재시도 로직 등 MLOps 코드 곳곳에서 보게 됩니다.

## 데코레이터의 정체: 함수를 인자로 받아 함수를 반환하는 함수

`@decorator` 문법은 `func = decorator(func)`의 문법 설탕(syntactic sugar)일 뿐입니다.

In [1]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("noni"))

# 위 @shout는 아래와 완전히 동일
def greet2(name):
    return f"hi, {name}"
greet2 = shout(greet2)
print(greet2("noni"))


HELLO, NONI
HI, NONI


## functools.wraps: 메타데이터 보존

데코레이터를 그냥 쓰면 원본 함수의 `__name__`, `__doc__`이 wrapper 것으로 덮어써집니다. `@wraps`로 보존해야 디버깅/문서화가 편합니다.

In [2]:
import functools

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        import time
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[{func.__name__}] {elapsed*1000:.3f}ms")
        return result
    return wrapper

@timed
def slow_square(n):
    """n의 제곱을 계산 (일부러 느리게)"""
    total = 0
    for _ in range(n * 100000):
        total += 1
    return n ** 2

print(slow_square(10))
print(slow_square.__name__, "-", slow_square.__doc__)  # wraps 덕분에 원본 정보 유지


[slow_square] 23.480ms
100
slow_square - n의 제곱을 계산 (일부러 느리게)


## 인자를 받는 데코레이터

데코레이터에 설정값을 넘기려면 함수를 한 겹 더 감싸야 합니다 (팩토리 패턴).

In [3]:
import functools
import time

def retry(times=3, delay=0.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"시도 {attempt} 실패: {e}")
                    time.sleep(delay)
            raise last_error
        return wrapper
    return decorator

counter = {"n": 0}

@retry(times=3)
def flaky():
    counter["n"] += 1
    if counter["n"] < 3:
        raise ValueError("아직 준비 안 됨")
    return "성공!"

print(flaky())


시도 1 실패: 아직 준비 안 됨
시도 2 실패: 아직 준비 안 됨
성공!


## 실전 예시: 결과 캐싱 (functools.lru_cache)

직접 구현할 수도 있지만, 표준 라이브러리가 이미 잘 만든 데코레이터를 제공합니다.

In [4]:
import functools

@functools.lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())  # 캐시 적중률 확인


832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


## built-in 데코레이터 미리보기: @staticmethod / @classmethod / @property

클래스 안에서 쓰는 데코레이터들이며, 06번 OOP 노트북에서 자세히 다룹니다.

In [5]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):  # 메서드인데 속성처럼 접근 (circle.area, 괄호 없이)
        return 3.14159 * self._radius ** 2

    @staticmethod
    def unit_circle():  # self 없이 호출되는, 클래스에 속한 일반 함수
        return Circle(1)

c = Circle(2)
print(c.area)          # 괄호 없이 호출
print(Circle.unit_circle().area)


12.56636
3.14159


## 헷갈렸던 점 / 더 찾아볼 것

- 데코레이터를 여러 개 겹쳐 쓸 때(`@a` `@b` 순서로 쌓음) 적용 순서는 **아래에서 위로**
- 클래스 기반 데코레이터(`__call__` 구현)도 있는데, 상태를 갖는 데코레이터가 필요할 때 유용